# 1.1 · Primera llamada

**Objetivo:** dejar el entorno listo y saber leer la respuesta completa, no solo el texto.

**Al terminar tendrás:** una primera respuesta de Claude inspeccionada pieza a pieza (bloques, `stop_reason`, `usage`) y la diferencia entre `system` y `messages`.

> Antes de empezar: el kernel debe ser `.venv` (arriba a la derecha). Si `00-comprobar-entorno.ipynb` no termina todo en ✅, vuelve a él.

## Preparación

Esta celda se repite al principio de todos los notebooks del taller:

- `load_dotenv()` carga la API key desde `.env`. **Nunca** se escribe la key en una celda.
- `sys.path.append("../..")` permite importar `config.py` desde la raíz del repositorio. Ahí está `MODEL`, el identificador del modelo que usamos en todo el taller: si se cambia en ese fichero, cambia en todos los notebooks.
- `anthropic.Anthropic()` crea el cliente. Lee `ANTHROPIC_API_KEY` del entorno y gestiona la autenticación, los reintentos y los tiempos de espera. Cada llamada que hagamos con él es una petición HTTP independiente a la API de Anthropic.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import anthropic
client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

In [ ]:
MODEL

## La llamada mínima

`client.messages.create` envía una conversación al modelo y devuelve su respuesta. Tiene tres parámetros obligatorios:

| Parámetro | Qué es |
|---|---|
| `model` | qué modelo responde |
| `max_tokens` | el número máximo de tokens que puede **generar** el modelo en esta respuesta. Es un tope de seguridad, no una longitud objetivo: si la respuesta necesita menos, usa menos |
| `messages` | la conversación: una lista de mensajes, cada uno con un `role` y un `content` |

**Tokens.** El modelo no trabaja con letras ni con palabras, sino con *tokens*: trozos de texto de unos pocos caracteres. Como orientación, una palabra en español son 1-2 tokens. Todo se mide en tokens: lo que cabe en una petición, lo que tarda en responder y lo que se paga.

**Roles.** En `messages` hay dos roles: `user` (lo que escribe la persona) y `assistant` (lo que respondió el modelo). Aquí enviamos un único mensaje `user`.

La última expresión de la celda es `response`, así que Jupyter muestra el objeto **entero**. Míralo antes de quedarte solo con el texto.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": "Explica en una frase qué es una incidencia de soporte.",
    }],
)
response

## Inspeccionar la respuesta

La respuesta es un objeto `Message` con `role="assistant"`: tiene la misma forma que los mensajes que enviamos, más algunos datos sobre la llamada (`id`, `model`…). Las tres partes importantes son:

| Campo | Responde a… |
|---|---|
| `content` | ¿qué ha dicho? |
| `stop_reason` | ¿por qué ha parado? |
| `usage` | ¿cuánto ha costado? |

Una celda por concepto.

### `content`: una lista de bloques

`content` **no es un string**: es una lista de bloques y cada bloque tiene un `type` que dice qué contiene. Es una lista porque una misma respuesta puede combinar varias cosas: por ejemplo, un bloque con el razonamiento del modelo y otro con la respuesta.

En esta respuesta verás un bloque `text` con la respuesta en lenguaje natural.

In [ ]:
response.content        # lista de bloques, no un string

Los tipos de los bloques, de un vistazo:

In [ ]:
[b.type for b in response.content]

### `stop_reason`: por qué paró

El modelo genera tokens hasta que algo lo detiene. `stop_reason` dice qué fue:

| Valor | Significa |
|---|---|
| `end_turn` | el modelo terminó su respuesta por su cuenta. Es el caso normal. |
| `max_tokens` | alcanzó el tope de `max_tokens`. La respuesta está **cortada a mitad**, no resumida: el modelo no sabe que tiene un límite y deja de escribir donde le pilla. |
| `tool_use` | el modelo se ha detenido para **pedir** que se ejecute una función. Solo ocurre cuando, en la llamada, le describimos funciones que puede pedir (*herramientas*); en esta llamada no lo hemos hecho. |

Por eso conviene mirar siempre `stop_reason` antes de usar la respuesta: con `end_turn` está completa; con `max_tokens`, no.

In [ ]:
response.stop_reason    # por qué paró

### `usage`: lo que se factura

La API se paga por tokens, con un precio por millón de tokens distinto para entrada y salida:

- `input_tokens`: los tokens que **enviamos**: el system prompt (lo vemos más abajo) y los mensajes.
- `output_tokens`: los tokens que **generó** el modelo, incluido su razonamiento si lo hubo. Los de salida son más caros que los de entrada.

Verás también campos `cache_creation_input_tokens` y `cache_read_input_tokens` a cero. Son de la *caché de prompts*: si se activa, la API puede guardar el principio de una petición que se repite mucho y cobrarlo más barato en las siguientes. Aquí no la hemos activado, así que están a cero.

In [ ]:
response.usage          # tokens de entrada y salida

In [ ]:
u = response.usage
print(f"entrada={u.input_tokens}  salida={u.output_tokens}")

### Extraer el texto

Además de `text`, en `content` puede aparecer un bloque `thinking`. Es el razonamiento del modelo: antes de responder, puede "pensar" el problema paso a paso y ese razonamiento llega como un bloque aparte, delante del texto. El modelo decide cuánto razonar según la pregunta: en una pregunta sencilla como esta puede no aparecer; en una más compleja, sí. Ese razonamiento cuenta como tokens de salida.

Por eso **no** se usa `response.content[0].text`: si el primer bloque es `thinking`, no tiene `.text` y la celda falla. Se busca el primer bloque de tipo `text`:

In [ ]:
texto = next(b.text for b in response.content if b.type == "text")
print(texto)

## El system prompt

Hasta ahora el modelo responde "en general". El parámetro `system` sirve para darle un papel:

- `system` define **quién es** el modelo: rol, tono, idioma, formato, límites.
- `messages` es **la conversación**: lo que dice el usuario y lo que responde el modelo.

No es un mensaje más de la conversación: va en un parámetro aparte, el modelo lo trata como las instrucciones que enmarcan todo lo demás y se envía **en cada llamada** (cuenta en `input_tokens`). En una aplicación real es donde van las reglas del negocio: qué puede y qué no puede hacer el asistente, cómo debe responder, qué hacer si no sabe algo.

Lo definimos en su propia celda para poder cambiarlo y reejecutar.

In [ ]:
SYSTEM = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, en tono cercano y como máximo en dos frases."
)

La misma pregunta, ahora con `system=SYSTEM`. Compara esta salida con la de arriba: mismo modelo y misma pregunta, pero cambian el tono y la longitud.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM,
    messages=[{
        "role": "user",
        "content": "Explica en una frase qué es una incidencia de soporte.",
    }],
)
print(next(b.text for b in response.content if b.type == "text"))

El system prompt también se ve en `usage`: ahora `input_tokens` es mayor, porque el system se ha enviado junto con la pregunta.

In [ ]:
response.usage.input_tokens

## Resumen

- La respuesta es un objeto con tres partes: **`content`** (lista de bloques), **`stop_reason`** (por qué paró) y **`usage`** (cuántos tokens se han consumido).
- El texto se extrae filtrando los bloques por `type == "text"`, porque puede haber otros bloques, como `thinking`.
- `stop_reason` dice si la respuesta está completa (`end_turn`) o cortada (`max_tokens`).
- `system` = quién es el modelo; `messages` = la conversación. El system se envía y se paga en cada llamada.

**Para el siguiente vídeo:** ¿qué pasa si en otra celda le pregunto por lo que le acabo de decir?

### Errores habituales

- **Kernel equivocado:** si no es `.venv`, falla `import anthropic` o se usan otras versiones. *Select Kernel → Python Environments → `.venv`*.
- **Tratar `response.content` como string:** es una lista de bloques.
- **Usar `response.content[0].text`:** falla si el primer bloque es `thinking`.
- **Hardcodear la API key en una celda:** acaba subida a git junto con el notebook. Siempre en `.env`.